In [1]:
# Import necessary libraries
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, ArrayType, FloatType

from pyspark.sql.functions import to_date, col, sum, count, date_format, when, udf
import glob
from geopy.geocoders import Nominatim
import time

### Initialize Spark Session

In [2]:
# Create a Spark session for distributed data processing
spark = SparkSession.builder \
      .master("local[1]") \
      .appName("toronto_parking_tickets") \
      .getOrCreate()

# Define the output folder for CSV exports
EXPORT_FOLDER = "./export"

### Helper Functions

In [3]:
# Function to format time from integer (e.g., 345 → "03:45")
def format_time(time):
    if time is None or time > 2359:
        return "NA"
    time_str = str(time).zfill(4)  # Ensure the string is always 4 digits
    return f"{time_str[:2]}:{time_str[2:]}"

# Dictionary mapping weekday numbers (1-7) to names
daydict = {1: "Sunday", 2: "Monday", 3: "Tuesday", 4: "Wednesday",
           5: "Thursday", 6: "Friday", 7: "Saturday"}

# Define User-Defined Functions (UDFs) for Spark transformations
getDayUDF = udf(lambda x: daydict[x], StringType())
formatTimeUDF = udf(format_time, StringType())

### Load Data

In [4]:
# Load all CSV files from the directory into a single DataFrame
csv_files = glob.glob("./parking-tickets-2022/*.csv")
df = spark.read.csv(csv_files, header=True, inferSchema=True)

### Data Cleaning & Preprocessing

In [5]:
# Rename columns for better readability
df = df.withColumnRenamed('location1','Proximity') \
       .withColumnRenamed('location2','Address') \
       .withColumnRenamed('location3','Proximity_optional') \
       .withColumnRenamed('location4','Address_optional') \
       .withColumnRenamed('tag_number_masked','Tag') \
       .withColumnRenamed('date_of_infraction','Issue_Date') \
       .withColumnRenamed('time_of_infraction','Issue Time') \
       .withColumnRenamed('infraction_code','Code') \
       .withColumnRenamed('infraction_description','Description') \
       .withColumnRenamed('set_fine_amount','Fine Amount') \
       .withColumnRenamed('province','Province')

# Convert date column to proper format and apply time formatting
df = df.withColumn('Issue_Date', to_date(df["Issue_Date"], "yyyyMMdd")) \
       .withColumn('Issue Time', formatTimeUDF(col('Issue Time'))) \
       .orderBy(col("Issue_Date"), col("Issue Time"))

### Aggregate Data

In [6]:
# Aggregate Total Fines Per Day
agg_fine_per_day = df.groupBy('Issue_Date').agg(sum('Fine Amount').alias('Total Fine'))
agg_fine_per_day.orderBy('Issue_Date').toPandas().to_csv(f"{EXPORT_FOLDER}/agg_fine_per_day.csv", index=False)

In [7]:
# Aggregate Tickets by Province
agg_prov = df.groupBy('Province').agg(count('Province').alias('Count'))
agg_prov.orderBy(col('Count').desc()).toPandas().to_csv(f"{EXPORT_FOLDER}/agg_prov.csv", index=False)

In [8]:
# Aggregate Tickets by Address
agg_address = df.groupBy('Address').agg(count('Address').alias('Count'))
agg_address.orderBy(col('Count').desc()).toPandas().to_csv(f"{EXPORT_FOLDER}/agg_address.csv", index=False)

In [9]:
# Extract distinct infraction codes & descriptions
infraction_codes = df.select('Code', 'Description').distinct()
infraction_codes = infraction_codes.orderBy('Code').dropDuplicates(subset=['Code'])

# Convert infraction codes into a dictionary for easy lookup
codedict = infraction_codes.toPandas().set_index('Code').T.to_dict('list')

# Define a UDF to retrieve the description based on the violation code
getCodeDescUDF = udf(lambda x: codedict[x][0] if x in codedict else "Unknown", StringType())

In [10]:
# Aggregate Tickets by Infraction Code
agg_code = df.groupBy('Code').agg(count('Code').alias('Count'))
agg_code = agg_code.withColumn('Description', getCodeDescUDF(col('Code')))
agg_code.orderBy('Code').toPandas().to_csv(f"{EXPORT_FOLDER}/agg_code.csv", index=False)

In [11]:
# Add and Aggregate Tickets by Day of the Week
df = df.withColumn('Day_of_the_Week', date_format(col("Issue_Date"), "EEEE"))
agg_dayofweek = df.groupBy('Day_of_the_Week').agg(count('*').alias('Count'))
agg_dayofweek.toPandas().to_csv(f"{EXPORT_FOLDER}/agg_dayofweek.csv", index=False)

In [12]:
# Aggregate Tickets by Month
df = df.withColumn('Month', date_format('Issue_Date', "MMMM"))
agg_month = df.groupBy('Month').agg(count('*').alias('Count'))
agg_month.toPandas().to_csv(f"{EXPORT_FOLDER}/agg_month.csv", index=False)

In [13]:
# Aggregate Tickets by Hour
df = df.withColumn(
    'Hour',
    when(col("Issue Time") == "NA", None)
    .otherwise(col("Issue Time").substr(1, 2).cast("int"))
)
agg_hour = df.groupBy("Hour").agg(count('*').alias("Count"))
agg_hour.orderBy(col("Hour").asc()).toPandas().to_csv(f"{EXPORT_FOLDER}/agg_hour.csv", index=False)

### Partitioned Dataset Exports

In [14]:
df.write.format('csv').option("header", "true").mode("ignore").partitionBy("Issue_Date").save("./spark-warehouse/bydate")

In [15]:
df.write.format('csv').option("header", "true").mode("ignore").partitionBy("Day_of_the_Week").save("./spark-warehouse/bydayofweek")

In [16]:
df.write.format('csv').option("header", "true").mode("ignore").partitionBy("Month").save("./spark-warehouse/bymonth")

In [17]:
df.write.format('csv').option("header", "true").mode("ignore").partitionBy("Code").save("./spark-warehouse/bycode")

In [18]:
df.write.format('csv').option("header", "true").mode("ignore").partitionBy("Province").save("./spark-warehouse/byprov")

### Geocoding

In [19]:
geolocator = Nominatim(user_agent="toronto_parking_data", timeout=5)
geo_cache = {}
provinces_list = agg_prov.select("Province").rdd.flatMap(lambda x: x).collect()
ca_prov = ['ON', 'QC', 'NS', 'NB', 'MB', 'BC', 'PE', 'SK', 'AB', 'NL', 'NT', 'YT', 'NU']


In [20]:
def get_coordinates(state_code):
    if state_code in geo_cache:
        return

    time.sleep(1.5)  # Prevent API rate limits
    
    if state_code in ca_prov:
        location = geolocator.geocode(f"{state_code}, Canada", country_codes="ca")
    else:
        location = geolocator.geocode(f"{state_code}, USA", country_codes="us")
        while not location:
            print(f"retrying search for {state_code}")
            time.sleep(1.5)  # Prevent API rate limits
            location = geolocator.geocode(f"{state_code}, USA", country_codes='us')
    geo_cache[state_code] = [location.latitude, location.longitude]

get_coordinates_udf = udf(lambda x: geo_cache[x], ArrayType(FloatType()))

In [21]:
for f in provinces_list:
    get_coordinates(f)

In [22]:
# Save Aggregate Tickets by Province with coordinates
agg_prov_with_coords = agg_prov.withColumn("Latitude", get_coordinates_udf(col("Province"))[0]) \
                               .withColumn("Longitude", get_coordinates_udf(col("Province"))[1])
agg_prov_with_coords.orderBy(col('Count').desc()).toPandas().to_csv(f"{EXPORT_FOLDER}/agg_prov_w_coord.csv", index=False)

In [23]:
df_add_coords = agg_address.orderBy(col("Count").desc()).limit(25)
add_cache = {}
add_list = df_add_coords.select("Address").rdd.flatMap(lambda x: x).collect()

In [24]:
def get_coordinates_add(add):
    if add in add_cache:
        return

    time.sleep(1.5)  # Prevent API rate limits
    
    location = geolocator.geocode(f"{add}, Toronto, Ontario, Canada", country_codes="ca")
    while not location:
        print(f"retrying search for {add}")
        time.sleep(1.5)  # Prevent API rate limits
        location = geolocator.geocode(f"{add}, Toronto, Ontario, Canada", country_codes="ca")
    add_cache[add] = [location.latitude, location.longitude]

get_coordinates_add_udf = udf(lambda x: add_cache[x], ArrayType(FloatType()))

In [25]:
for f in add_list:
    get_coordinates_add(f)

In [26]:
# Save Aggregate Tickets by Address with coordinates
agg_add_with_coords = df_add_coords.withColumn("Latitude", get_coordinates_add_udf(col("Address"))[0]) \
                          .withColumn("Longitude", get_coordinates_add_udf(col("Address"))[1])
agg_add_with_coords.orderBy(col('Count').desc()).toPandas().to_csv(f"{EXPORT_FOLDER}/agg_add_w_coord.csv", index=False)

### Save Processed Dataset 

In [ ]:
# save fully processed dataframe
df.toPandas().to_csv(f"{EXPORT_FOLDER}/processed_dataset.csv", index=False) 